## 1. Install dependencies
Installs the packages needed for the rest of the notebook: `datasets`, `huggingface_hub`, `pandas`, `openai`, `scikit-learn`.

In [2]:
!pip install -q datasets huggingface_hub pandas openai scikit-learn

## 2. Load the non-SNV variant dataset
Loads `wanglab/variant_effect_non_snv` from Hugging Face and inspects the test split's structure.

In [3]:
from datasets import load_dataset

ds = load_dataset("wanglab/variant_effect_non_snv")
print(ds)

test_df = ds['test'].to_pandas()
print("\nColumns:", test_df.columns.tolist())
print("\nTotal test rows:", len(test_df))

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'reference_sequence', 'mutated_sequence', 'cleaned_pathogenicity', '__index_level_0__'],
        num_rows: 35215
    })
    test: Dataset({
        features: ['question', 'answer', 'reference_sequence', 'mutated_sequence', 'cleaned_pathogenicity', '__index_level_0__'],
        num_rows: 873
    })
})

Columns: ['question', 'answer', 'reference_sequence', 'mutated_sequence', 'cleaned_pathogenicity', '__index_level_0__']

Total test rows: 873


## 3. Build a small balanced sample
Assembles a 30-row sample (15 pathogenic + 15 benign) to prototype the classification approach on before scaling up.

In [4]:
import pandas as pd

# Build a small, balanced sample (15 pathogenic + 15 benign) to work with
sample_df = pd.concat([
    test_df[test_df['cleaned_pathogenicity'] == 'pathogenic'].sample(15, random_state=42),
    test_df[test_df['cleaned_pathogenicity'] == 'benign'].sample(15, random_state=42)
]).reset_index(drop=True)

sample_df['true_label'] = sample_df['cleaned_pathogenicity'].str.capitalize()

print(sample_df[['question', 'true_label']])
print("\nSample size:", len(sample_df))

                                             question  true_label
0   A mutation at chromosome position 108734572 on...  Pathogenic
1   Mutation at chromosome 15, position 57232389, ...  Pathogenic
2   Chromosome 1, position 36091720, gene ADPRS (A...  Pathogenic
3   Mutation at chromosome 7, position 99392785, w...  Pathogenic
4   Variant at chromosome 3, position 48892052, ge...  Pathogenic
5   A mutation at chromosome position 97584819 on ...  Pathogenic
6   A genetic variant on chromosome X, position 11...  Pathogenic
7   Evaluate the clinical significance of the muta...  Pathogenic
8   Benign or pathogenic: chromosome 4, position 4...  Pathogenic
9   Mutation at chromosome 5, position 39311206, w...  Pathogenic
10  For chromosome 1, position 52976681, gene SCP2...  Pathogenic
11  Gene FBXO7 (F-box protein 7) variant at chromo...  Pathogenic
12  Does the variant impacting SLC25A20 (solute ca...  Pathogenic
13  Is the genetic change at chromosome 16, positi...  Pathogenic
14  Assess

## 4. Connect to the local Ollama server
Creates an `OpenAI` client pointed at the local Ollama endpoint and lists the models available (`qwen3:4b`, `qwen3:1.7b`).

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  #ollam value ignore kore
)

# checking if ollama id available
models = client.models.list()
for m in models.data:
    print(m.id)

qwen3:4b
qwen3:1.7b


## 5. Define the classifier and smoke-test it
`classify_variant()` sends a prompt to a local Ollama model and returns its reasoning/content; run once on a single row to sanity-check the setup.

In [6]:
def classify_variant(question_text, model_name="qwen3:1b", max_retries=2):
    """Send one variant question to a local Ollama model, return reasoning + content."""
    prompt = f"""{question_text}

After your analysis, you MUST end with exactly this line and nothing else after it:
Final Answer: [Pathogenic|Benign]"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                max_tokens=1500
            )
            message = response.choices[0].message
            return {
                "reasoning": getattr(message, "reasoning", None),
                "content": message.content,
                "error": None
            }
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")

    return {"reasoning": None, "content": None, "error": "Failed after retries"}

# smoke test
test_result = classify_variant(sample_df.iloc[0]['question'], model_name="qwen3:1.7b")
print(test_result)

{'reasoning': 'Okay, let\'s tackle this question about the mutation at chromosome position 108734572 on chromosome X in gene IRS4. The user is asking whether this is benign or pathogenic, and if it\'s pathogenic, which diseases it\'s linked to.\n\nFirst, I need to recall what IRS4 is. IRS4 is a gene that codes for insulin receptor substrate 4, which is part of the insulin signaling pathway. This gene is involved in cellular processes related to insulin, such as metabolism and growth. \n\nNow, the position given is 108734572 on chromosome X. I should check if this is a known mutation site. I remember that mutations in IRS4 can lead to various disorders. For example, mutations in IRS4 are associated with certain types of cancer, particularly breast and prostate cancer. There\'s also a condition called insulin-like growth factor 2 (IGF2) mutations, but that\'s different. \n\nWait, the user is asking about IRS4 specifically. I need to confirm if this particular position is a known mutation

## 6. Define the answer parser
`parse_prediction()` extracts a `Pathogenic`/`Benign` label from the model's raw text output; tested against the previous cell's result.

In [7]:
import re

def parse_prediction(content):
    """Extract Pathogenic/Benign from the model's content field, tolerant of brackets/extra text."""
    if not content:
        return "Uncertain"
    match = re.search(r"Final Answer:\s*\[?(Pathogenic|Benign)", content, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Uncertain"

# output er against e check
print(parse_prediction(test_result['content']))

Pathogenic


## 7. Label the full test set
Adds a `true_label` column to the entire 873-row test set ahead of a full run.

In [8]:
test_df['true_label'] = test_df['cleaned_pathogenicity'].str.capitalize()
print(test_df['true_label'].value_counts())

true_label
Pathogenic    523
Benign        350
Name: count, dtype: int64


## 8. Run batch inference on the full test set (qwen3:1.7b)
Classifies all 873 test rows with `qwen3:1.7b`, writing results to `results_ve_non_snv_full_1p7b.jsonl` (resumable by row index).

In [9]:
import json
import os

output_file = "results_ve_non_snv_full_1p7b.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in test_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 25 == 0:
        print(f"Processing {idx+1}/{len(test_df)}...")

    result = classify_variant(row['question'], model_name="qwen3:1.7b")

    record = {
        "row_index": idx,
        "true_label": row['true_label'],
        "reasoning": result['reasoning'],
        "content": result['content'],
        "error": result['error']
    }

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done")

Already completed: 873 rows
Done


## 9. Load and parse the full-test results
Loads the 1.7b results file, applies `parse_prediction`, and checks the prediction distribution and error count.

In [10]:
import json
import pandas as pd

results = []
with open("results_ve_non_snv_full_1p7b.jsonl", 'r') as f:
    for line in f:
        results.append(json.loads(line))

results_df = pd.DataFrame(results)
results_df['predicted_label'] = results_df['content'].apply(parse_prediction)

print("Total rows:", len(results_df))
print("\nPrediction distribution:")
print(results_df['predicted_label'].value_counts())
print("\nErrors:", results_df['error'].notna().sum())

Total rows: 873

Prediction distribution:
predicted_label
Pathogenic    768
Benign         95
Uncertain      10
Name: count, dtype: int64

Errors: 0


## 10. Evaluate qwen3:1.7b on the full test set
Computes accuracy, macro precision/recall/F1, and a full classification report.

In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

y_true = results_df['true_label']
y_pred = results_df['predicted_label']

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=['Pathogenic', 'Benign'], average='macro', zero_division=0
)

print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")

print("\nClassification report ")
print(classification_report(y_true, y_pred, labels=['Pathogenic', 'Benign'], zero_division=0))

Accuracy:  58.42%
Precision: 53.50%
Recall:    50.84%
F1-Score:  45.98%

Classification report 
              precision    recall  f1-score   support

  Pathogenic       0.61      0.89      0.72       523
      Benign       0.46      0.13      0.20       350

   micro avg       0.59      0.58      0.59       873
   macro avg       0.53      0.51      0.46       873
weighted avg       0.55      0.58      0.51       873



## 11. Break down the Pathogenic-class metrics
Reports per-class F1, precision, and recall, isolating how well the model does specifically on the Pathogenic label.

In [12]:
from sklearn.metrics import f1_score, precision_score, recall_score

f1_scores = f1_score(y_true, y_pred, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
precision_scores = precision_score(y_true, y_pred, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
recall_scores = recall_score(y_true, y_pred, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)

# 1 pathgenic 0 benign
print(f"Pathogenic-class F1:        {f1_scores[0]*100:.2f}%")
print(f"Pathogenic-class Precision: {precision_scores[0]*100:.2f}%")
print(f"Pathogenic-class Recall:    {recall_scores[0]*100:.2f}%")

Pathogenic-class F1:        72.19%
Pathogenic-class Precision: 60.68%
Pathogenic-class Recall:    89.10%


## 12. Sample 200 stratified rows for the 4b comparison
Draws a smaller stratified sample so the larger `qwen3:4b` model can be evaluated without running the full 873-row set.

In [13]:
from sklearn.model_selection import train_test_split

sample_200_df, _ = train_test_split(
    test_df,
    train_size=200,
    stratify=test_df['true_label'],
    random_state=42
)
sample_200_df = sample_200_df.reset_index(drop=True)

print(sample_200_df['true_label'].value_counts())
print("Sample size:", len(sample_200_df))

true_label
Pathogenic    120
Benign         80
Name: count, dtype: int64
Sample size: 200


## 13. Run batch inference on the 200-row sample (qwen3:4b)
Classifies the 200-row sample with `qwen3:4b` using the original `classify_variant`, writing to `results_ve_non_snv_200_4b.jsonl`.

In [14]:
import json
import os

output_file = "results_ve_non_snv_200_4b.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in sample_200_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 20 == 0:
        print(f"Processing {idx+1}/{len(sample_200_df)}...")

    result = classify_variant(row['question'], model_name="qwen3:4b")

    record = {
        "row_index": idx,
        "true_label": row['true_label'],
        "reasoning": result['reasoning'],
        "content": result['content'],
        "error": result['error']
    }

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done")

Already completed: 200 rows
Done


## 14. Load and parse the 4b results
Reveals a problem: most predictions come back "Uncertain" instead of a clean label.

In [15]:
import json
import pandas as pd

results_4b = []
with open("results_ve_non_snv_200_4b.jsonl", 'r') as f:
    for line in f:
        results_4b.append(json.loads(line))

results_4b_df = pd.DataFrame(results_4b)
results_4b_df['predicted_label'] = results_4b_df['content'].apply(parse_prediction)

print("Total rows:", len(results_4b_df))
print("\nPrediction distribution:")
print(results_4b_df['predicted_label'].value_counts())
print("\nErrors:", results_4b_df['error'].notna().sum())

Total rows: 200

Prediction distribution:
predicted_label
Uncertain    194
Benign         6
Name: count, dtype: int64

Errors: 0


## 15. Diagnose the "Uncertain" results
Inspects the raw content/reasoning for the first few rows — content is empty, meaning the model's answer got truncated before the "Final Answer" line (max_tokens too low for 4b's verbose reasoning).

In [16]:
for i in range(5):
    print(f"--- Row {i} ---")
    print("Content:", repr(results_4b_df.iloc[i]['content']))
    print("Reasoning (last 300 chars):", repr(results_4b_df.iloc[i]['reasoning'])[-300:] if results_4b_df.iloc[i]['reasoning'] else None)
    print("Error:", results_4b_df.iloc[i]['error'])
    print()

--- Row 0 ---
Content: ''
Reasoning (last 300 chars): hout variant type is not sufficient.\n\nBut note: The problem might be testing if we know that SOX4 variants are typically benign.\n\nI found a reference: \n- In the context of cancer, SOX4 is a tumor suppressor gene. Variants in SOX4 can be associated with cancer. However, the specific position 21'
Error: None

--- Row 1 ---
Content: ''
Reasoning (last 300 chars): ot pathogenic? Or benign.\n\nBut note: The problem says "this variant", implying it\'s a specific variant that is known. However, without the change, we cannot evaluate.\n\nI recall that in some contexts, the position 42731193 might be the reference for a known variant? \n\nLet me check the ClinVar'
Error: None

--- Row 2 ---
Content: ''
Reasoning (last 300 chars): ogenic.\n\nGiven that the problem asks for a final answer of either "Pathogenic" or "Benign", and the context of RMRP, I think it\'s safe to say that if the variant is in this gene and it is a known pathogenic 

## 16. Define an improved classifier for 4b
`classify_variant_v2()` raises `max_tokens` and sets an explicit context window to stop the 4b model's answers from being cut off.

In [17]:
def classify_variant_v2(question_text, model_name="qwen3:4b", max_retries=2, max_tokens=4000):
    """v2: higher max_tokens + explicit context window to prevent silent truncation on verbose models."""
    prompt = f"""{question_text}

After your analysis, you MUST end with exactly this line and nothing else after it:
Final Answer: [Pathogenic|Benign]"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                max_tokens=max_tokens,
                extra_body={"options": {"num_ctx": 8192}}
            )
            message = response.choices[0].message
            return {
                "reasoning": getattr(message, "reasoning", None),
                "content": message.content,
                "error": None
            }
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")

    return {"reasoning": None, "content": None, "error": "Failed after retries"}

## 17. Time a 10-row test with the fixed classifier
Runs v2 on 10 rows to confirm it resolves the truncation issue and to estimate how long the full 200-row run will take.

In [18]:
import time

test_10_df = sample_200_df.head(10)

start_time = time.time()

for idx, row in test_10_df.iterrows():
    row_start = time.time()
    result = classify_variant_v2(row['question'], model_name="qwen3:4b")
    row_time = time.time() - row_start

    predicted = parse_prediction(result['content'])
    status = "OK" if predicted != "Uncertain" else "STILL UNCERTAIN"

    print(f"Row {idx}: {row_time:.1f}s | True: {row['true_label']} | Predicted: {predicted} | {status}")

total_time = time.time() - start_time
print(f"\nTotal: {total_time:.1f}s for 10 rows ({total_time/10:.1f}s/row average)")
print(f"Estimated time for 200 rows: {(total_time/10*200)/60:.1f} minutes")

Row 0: 45.2s | True: Benign | Predicted: Benign | OK
Row 1: 29.7s | True: Benign | Predicted: Benign | OK
Row 2: 33.0s | True: Pathogenic | Predicted: Pathogenic | OK
Row 3: 30.7s | True: Pathogenic | Predicted: Benign | OK
Row 4: 33.5s | True: Benign | Predicted: Pathogenic | OK
Row 5: 47.5s | True: Pathogenic | Predicted: Pathogenic | OK
Row 6: 46.0s | True: Pathogenic | Predicted: Uncertain | STILL UNCERTAIN
Row 7: 46.5s | True: Pathogenic | Predicted: Uncertain | STILL UNCERTAIN
Row 8: 24.0s | True: Pathogenic | Predicted: Benign | OK
Row 9: 45.8s | True: Benign | Predicted: Uncertain | STILL UNCERTAIN

Total: 381.8s for 10 rows (38.2s/row average)
Estimated time for 200 rows: 127.3 minutes


## 18. Run batch inference on the 200-row sample (qwen3:4b, fixed)
Classifies the full 200-row sample with `classify_variant_v2`, writing to `results_ve_non_snv_200_4b_v2.jsonl`.

In [19]:
import json
import os

output_file = "results_ve_non_snv_200_4b_v2.jsonl"

if not os.path.exists(output_file):
    open(output_file, 'w').close()

completed_indices = set()
with open(output_file, 'r') as f:
    for line in f:
        if line.strip():
            completed_indices.add(json.loads(line)['row_index'])

print(f"Already completed: {len(completed_indices)} rows")

for idx, row in sample_200_df.iterrows():
    if idx in completed_indices:
        continue

    if idx % 10 == 0:
        print(f"Processing {idx+1}/{len(sample_200_df)}...")

    result = classify_variant_v2(row['question'], model_name="qwen3:4b")

    record = {
        "row_index": idx,
        "true_label": row['true_label'],
        "reasoning": result['reasoning'],
        "content": result['content'],
        "error": result['error']
    }

    with open(output_file, 'a') as f:
        f.write(json.dumps(record) + "\n")

print("Done")

Already completed: 200 rows
Done


In [20]:
import json

with open("results_ve_non_snv_200_4b_v2.jsonl", 'r') as f:
    lines_4b_v2 = [json.loads(line) for line in f]

print("Total lines in file:", len(lines_4b_v2))

empty_count = sum(1 for l in lines_4b_v2 if not l['content'] and not l['reasoning'])
print(f"Rows with both content AND reasoning empty: {empty_count}")

error_count = sum(1 for l in lines_4b_v2 if l['error'])
print(f"Rows with an error recorded: {error_count}")

print("\n--- First result ---")
print("Content:", lines_4b_v2[0]['content'])
print("Reasoning length:", len(lines_4b_v2[0]['reasoning']) if lines_4b_v2[0]['reasoning'] else 0)

if len(lines_4b_v2) != 200:
    print("\nWARNING rerun neededf")

Total lines in file: 200
Rows with both content AND reasoning empty: 0
Rows with an error recorded: 0

--- First result ---
Content: The variant described is at chromosome 6, position 21595402, affecting the gene SOX4 (SRY-box transcription factor 4). After analysis using standard genomic coordinates (GRCh38), the gene SOX4 is located at approximately 6:13370000-13400000. The position 21595402 is outside this region and does not fall within the SOX4 gene interval. Therefore, this variant does not affect the SOX4 gene as stated. Additionally, no pathogenic variants for SOX4 are documented at this specific position in major databases such as ClinVar or HGMD. Variants in SOX4 are typically associated with benign conditions or non-pathogenic variations, and there are no well-established pathogenic variants at this position linked to specific illnesses. Given the evidence, the variant is classified as benign.

Final Answer: Benign
Reasoning length: 9307


In [21]:
import re
import pandas as pd

def parse_prediction_v3(content, reasoning):
    """Try content first; fall back to scanning reasoning for a final verdict if content is empty/Uncertain.
    (?!\|) rejects a match immediately followed by '|', which only happens when the model
    copied the unfilled '[Pathogenic|Benign]' template rather than choosing one.
    """
    def _match(text):
        if not text:
            return None
        m = re.search(r"Final Answer:\s*\[?(Pathogenic|Benign)(?!\|)", text, re.IGNORECASE)
        return m.group(1).capitalize() if m else None

    result = _match(content)
    if result:
        return result

    if reasoning:
        matches = re.findall(r"Final Answer:\s*\[?(Pathogenic|Benign)(?!\|)\]?", reasoning, re.IGNORECASE)
        if matches:
            return matches[-1].capitalize()

    return "Uncertain"

results_4b_v2_df = pd.DataFrame(lines_4b_v2)
results_4b_v2_df['predicted_label'] = results_4b_v2_df.apply(
    lambda row: parse_prediction_v3(row['content'], row['reasoning']), axis=1
)

print(results_4b_v2_df['predicted_label'].value_counts())

predicted_label
Benign        109
Pathogenic     91
Name: count, dtype: int64


In [22]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, f1_score, precision_score, recall_score

y_true_4b = results_4b_v2_df['true_label']
y_pred_4b = results_4b_v2_df['predicted_label']

accuracy_4b = accuracy_score(y_true_4b, y_pred_4b)
precision_4b, recall_4b, f1_4b, _ = precision_recall_fscore_support(
    y_true_4b, y_pred_4b, labels=['Pathogenic', 'Benign'], average='macro', zero_division=0
)

f1_scores_4b = f1_score(y_true_4b, y_pred_4b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
precision_scores_4b = precision_score(y_true_4b, y_pred_4b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)
recall_scores_4b = recall_score(y_true_4b, y_pred_4b, labels=['Pathogenic', 'Benign'], average=None, zero_division=0)

print(f"Accuracy:  {accuracy_4b*100:.2f}%")
print(f"Macro Precision: {precision_4b*100:.2f}%")
print(f"Macro Recall:    {recall_4b*100:.2f}%")
print(f"Macro F1-Score:  {f1_4b*100:.2f}%")

print(f"\nPathogenic-class F1:        {f1_scores_4b[0]*100:.2f}%")
print(f"Pathogenic-class Precision: {precision_scores_4b[0]*100:.2f}%")
print(f"Pathogenic-class Recall:    {recall_scores_4b[0]*100:.2f}%")

print("\nClassification report")
print(classification_report(y_true_4b, y_pred_4b, labels=['Pathogenic', 'Benign'], zero_division=0))

Accuracy:  53.50%
Macro Precision: 54.44%
Macro Recall:    54.58%
Macro F1-Score:  53.36%

Pathogenic-class F1:        55.92%
Pathogenic-class Precision: 64.84%
Pathogenic-class Recall:    49.17%

Classification report
              precision    recall  f1-score   support

  Pathogenic       0.65      0.49      0.56       120
      Benign       0.44      0.60      0.51        80

    accuracy                           0.54       200
   macro avg       0.54      0.55      0.53       200
weighted avg       0.57      0.54      0.54       200

